<a href="https://colab.research.google.com/github/Lagnajit09/100x_AI_ML/blob/main/TrainingModel07_MultiLayerTransformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Multi Layer Trasformer**

The core distinction in one sentence

- Multi-head attention = multiple perspectives looking at the same information in parallel.
- Multi-layer stacking = the same mechanism applied repeatedly, each time on a more refined version of the information.

### Multi-head attention — recap with a clearer lens
Inside one block, you have (say) 8 heads. They all receive the same input at the same time. Each head learns its own W_Q, W_K, W_V — so each head learns to look for a different kind of relationship.
```
                    ┌─→ Head 1: "who modifies whom?"──────────┐
                    │                                         │
                    ├─→ Head 2: "subject-verb relationships?" ┤
Input token vectors ┤                                         ├─→ concat → output     │                                         │
                    ├─→ Head 3: "what's the topic?"           ┤
                    │                                         │
                    └─→ Head 8: "pronoun references?"─────────┘
                    
                    All 8 heads working in PARALLEL
                    All 8 heads see the SAME input
                    All 8 heads finish at the SAME time
```                    
Heads are about breadth at one moment — multiple angles on the current state of the data.

### Multi-layer stacking — what's actually different
You take a whole transformer block (attention + FFN + norms + residuals — the entire thing you built) and run it again on its own output. Then again. Then again.
```
Input embeddings
       │
       ▼
┌──────────────┐
│   Block 1    │  ← sees raw word meanings
│  (attention  │
│   + FFN)     │
└──────────────┘
       │
       ▼  ← Block 1's output, now an enriched version of the input
┌──────────────┐
│   Block 2    │  ← sees Block 1's refined representations
│  (attention  │
│   + FFN)     │
└──────────────┘
       │
       ▼  ← Block 2's output, refined again
┌──────────────┐
│   Block 3    │  ← sees Block 2's even more refined representations
│  ...         │
└──────────────┘
   ...
```
The crucial difference: each layer operates on the previous layer's output, not on the raw input.

### What research has shown each layer actually learns
This isn't speculation — researchers have probed real models and found patterns. In a 12-layer GPT-2:

- Layers 1–2: Surface features — word identity, simple positional patterns, basic part-of-speech-like distinctions
- Layers 3–5: Syntactic structures — which words modify which, basic phrase boundaries, subject-verb agreement
- Layers 6–9: Semantic relationships — coreference (who "she" refers to), topic tracking, entity relationships
- Layers 10–12: Task-specific patterns — completing the sentence sensibly, matching style, factual recall

In [ ]:
# Import Modules
import torch
import torch.nn as nn
import math

In [ ]:
# Existing Multi-Head Attention Class:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        seq_len = x.shape[0]
        Q = self.W_Q(x)
        K = self.W_K(x)
        V = self.W_V(x)
        Q = Q.view(seq_len, self.num_heads, self.d_k).permute(1, 0, 2)
        K = K.view(seq_len, self.num_heads, self.d_k).permute(1, 0, 2)
        V = V.view(seq_len, self.num_heads, self.d_k).permute(1, 0, 2)
        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_k)
        attention_weights = torch.softmax(scores, dim=-1)
        head_outputs = attention_weights @ V
        head_outputs = head_outputs.permute(1, 0, 2).contiguous().view(seq_len, self.d_model)
        return self.W_O(head_outputs), attention_weights

In [ ]:
# Existing Feed Forward Network:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.layer1 = nn.Linear(d_model, d_ff)
        self.relu   = nn.ReLU()
        self.layer2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.layer2(self.relu(self.layer1(x)))

In [ ]:
# Existing Transformer Block:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()
        self.attention    = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = FeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        attn_out, _ = self.attention(self.norm1(x))
        x = x + attn_out
        ff_out = self.feed_forward(self.norm2(x))
        x = x + ff_out
        return x

In [ ]:
# Positional Encoding:
def positional_encoding(seq_len, d_model):
    pe = torch.zeros(seq_len, d_model)
    for pos in range(seq_len):
        for i in range(0, d_model, 2):
            denom = 10000 ** (i / d_model)
            pe[pos, i]     = math.sin(pos / denom)
            pe[pos, i + 1] = math.cos(pos / denom)
    return pe

In [ ]:
class MiniLLM(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)

        # NEW: instead of ONE block, create a LIST of N blocks
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, num_heads, d_ff)
            for _ in range(num_layers)
        ])

        # Final layer norm before the output head (standard in GPT-style models)
        self.final_norm = nn.LayerNorm(d_model)

        self.output_head = nn.Linear(d_model, vocab_size)

    def forward(self, token_ids):
        seq_len = token_ids.shape[0]

        # Step 1: Token embeddings + positional encoding
        x = self.embedding(token_ids)
        x = x + positional_encoding(seq_len, x.shape[1])

        # Step 2: Pass through EACH block, one after another
        for block in self.blocks:
            x = block(x)

        # Step 3: Final normalization
        x = self.final_norm(x)

        # Step 4: Output head
        logits = self.output_head(x)

        return logits

In [ ]:
text = """
the quick brown fox jumps over the lazy dog
artificial intelligence is transforming industries across the world
machine learning models learn patterns from historical data
deep learning networks contain multiple layers of neurons
natural language processing enables communication between humans and machines
computer vision systems analyze images and videos
cloud platforms provide scalable infrastructure for applications
software developers write maintainable and efficient code
devops engineers automate deployment pipelines
kubernetes orchestrates containerized applications at scale
databases store and retrieve information efficiently
cybersecurity protects systems against threats and attacks
data scientists explore datasets to discover insights
backend services expose apis for client applications
frontend frameworks create interactive user interfaces
microservices architectures break applications into smaller services
distributed systems coordinate work across multiple machines
version control systems track changes in source code
testing frameworks ensure software quality and reliability
automation improves productivity and reduces errors
"""

In [ ]:
# ── Training (same as before, just with num_layers) ────────────────────

# text   = "i love code and i love ai"
tokens = text.lower().split()
vocab  = sorted(set(tokens))
word_to_id = {w: i for i, w in enumerate(vocab)}
id_to_word = {i: w for w, i in word_to_id.items()}
token_ids = torch.tensor([word_to_id[t] for t in tokens])

print("Number of tokens:", len(tokens))
print("Vocabulary size:", len(vocab))
print("\nVocabulary:")
print(vocab)
print("\nToken IDs:")
print(token_ids)

inputs  = token_ids[:-1]
targets = token_ids[1:]

# Config
vocab_size = len(vocab)
d_model    = 8
num_heads  = 2
d_ff       = 32
num_layers = 3            # ← NEW! 3 stacked transformer blocks

model     = MiniLLM(vocab_size, d_model, num_heads, d_ff, num_layers)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn   = nn.CrossEntropyLoss()

print(f"Model has {sum(p.numel() for p in model.parameters())} parameters")
print(f"Stack depth: {num_layers} transformer blocks\n")

for step in range(200):
    logits = model(inputs)
    loss   = loss_fn(logits, targets)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 20 == 0:
        print(f"Step {step:3d} | Loss: {loss.item():.4f}")

# Final predictions
print("\n=== Final Predictions ===")
with torch.no_grad():
    logits = model(inputs)
    predicted_ids = logits.argmax(dim=-1)

for i, (inp, pred, target) in enumerate(zip(inputs, predicted_ids, targets)):
    input_word  = id_to_word[inp.item()]
    pred_word   = id_to_word[pred.item()]
    target_word = id_to_word[target.item()]
    correct     = "✓" if pred.item() == target.item() else "✗"
    print(f"  [{correct}] Input: '{input_word}' → Predicted: '{pred_word}' (should be: '{target_word}')")

Number of tokens: 142
Vocabulary size: 118

Vocabulary:
['across', 'against', 'analyze', 'and', 'apis', 'applications', 'architectures', 'artificial', 'at', 'attacks', 'automate', 'automation', 'backend', 'between', 'break', 'brown', 'changes', 'client', 'cloud', 'code', 'communication', 'computer', 'contain', 'containerized', 'control', 'coordinate', 'create', 'cybersecurity', 'data', 'databases', 'datasets', 'deep', 'deployment', 'developers', 'devops', 'discover', 'distributed', 'dog', 'efficient', 'efficiently', 'enables', 'engineers', 'ensure', 'errors', 'explore', 'expose', 'for', 'fox', 'frameworks', 'from', 'frontend', 'historical', 'humans', 'images', 'improves', 'in', 'industries', 'information', 'infrastructure', 'insights', 'intelligence', 'interactive', 'interfaces', 'into', 'is', 'jumps', 'kubernetes', 'language', 'layers', 'lazy', 'learn', 'learning', 'machine', 'machines', 'maintainable', 'microservices', 'models', 'multiple', 'natural', 'networks', 'neurons', 'of', 'or

### Code Explanation:
`nn.ModuleList` is a special PyTorch container designed specifically for holding a list of `nn.Module` objects (layers, blocks, etc.). Why not just use a regular Python list?

Because PyTorch needs to find every parameter in your model for .`parameters()`, `.to(device)`, `.train()`, `.eval()`, `optimizer registration`, and saving/loading checkpoints. A regular Python list hides its contents from PyTorch — your model would technically work but the optimizer would never see those blocks' weights, and `loss.backward()` would update nothing in them.

`nn.ModuleList` looks and behaves like a Python list (you can index it, iterate over it, get its length), but it also registers every module inside as a proper submodule of the parent model. So when you do `model.parameters()`, PyTorch walks into the ModuleList and finds all the weights inside every block.

Quick rule: anytime you store `nn.Modules` in a collection, use `nn.ModuleList` (for lists) or `nn.ModuleDict` (for dicts). Never plain Python list/dict.

The list comprehension creates 3 fresh, independent TransformerBlock instances. Each one has its own `W_Q, W_K, W_V, W_O, FFN weights, and layer norms`. They're not sharing weights — they're 3 separate blocks that will learn different things.

`self.final_norm` is a standard addition in GPT-style models. After all the blocks have done their work, one final layer norm cleans up the output before it hits the output head. This isn't strictly necessary in tiny models, but it stabilizes training in real models and it's the canonical pattern.